# Suv oylari — v3 (fenologiya)

`ekin_yerlar_qisqa.gpkg` dan **`ekin_yerlar_qisqa_v3_fenologiya.gpkg`** yasaladi. Asl faylga tegilmaydi.

O'zgaradigan ustunlar — **faqat**:
- `suv_boshi`, `suv_oxiri` — barcha mavsumli konturlarda;
- `season` — faqat Ozuqa yoki Bug'doy ustidagi Paxta konturlarida (`Mavsumiy ekin` → `Ikkilamchi ekin`).

Geometriya va boshqa atributlar o'zgarmaydi (fayl nusxalanadi, nusxada SQL UPDATE; 6-qadam buni tekshiradi).

Qoidalar — `config/suv_oylari.yaml`:
1. **Birlamchi** — hammasi 3-oydan, tugashi ekin turi bo'yicha (3-5 yoki 3-6; Paxta ostidagi Ozuqa/Bug'doy 3-4).
2. **Ikkilamchi** — shu daladagi Birlamchi tugagan oydan keyin, 10-oygacha (3-5 → 6-10, 3-6 → 7-10).
3. **Mavsumiy** — 3-10.
4. **Ozuqa / Bug'doy ustidagi Paxta** — Ikkilamchi ekin, 5-10.

Bitta dala: ikki kontur kesishmasi ulardan birining maydonining ≥ 50% i.

**Muhit:** kernel — `fenologiya` conda muhiti. Birinchi marta (bir marta) PowerShell da:
```
& "$env:LOCALAPPDATA\miniconda3\Scripts\conda.exe" install -n fenologiya -c conda-forge ipykernel -y
```
so'ng VS Code da *Select Kernel → Python Environments → fenologiya*.

In [1]:
# 0. Sozlash — fayl yo'llari shu yerda
import os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, "src"))
import fenologiya.muhit  # noqa: F401  (DLL/GDAL — numpy dan oldin)
import pandas as pd
from fenologiya import suv

KIRISH = suv.standart_kirish()            # bir_ikki_lamchi/natija/ekin_yerlar_qisqa.gpkg
VERSIYA = "v3"
CHIQISH_PAPKA = os.path.dirname(KIRISH)    # natija asl fayl yonida
NOM = f"{os.path.splitext(os.path.basename(KIRISH))[0]}_{VERSIYA}_fenologiya"
CHIQISH = os.path.join(CHIQISH_PAPKA, NOM + ".gpkg")
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200)
print("Kirish :", KIRISH); print("Chiqish:", CHIQISH)

Kirish : d:\Tasks\bir_ikki_lamchi\natija\ekin_yerlar_qisqa_v2_fenologiya.gpkg
Chiqish: d:\Tasks\bir_ikki_lamchi\natija\ekin_yerlar_qisqa_v2_fenologiya_v3_fenologiya.gpkg


## 1. Qoidalar (config/suv_oylari.yaml)

In [2]:
cfg, zm = suv.sozlamalar()
print(open(os.path.join(ROOT, "config", "suv_oylari.yaml"), encoding="utf-8").read())

# Suv ichish oylari (05_suv_oylari.py) — [boshi, oxiri], ikkala oy ham kiradi.
# Asos: 04 NDVI natijasi (69 tuman, 2026), viloyat × zona × ekin turi; 2026-11-06 kelishilgan.
#
# 1) Birlamchi — hammasi 3-oydan (ekish oldi sug'orish ham kiradi); tugashi ekin turi bo'yicha (quyida).
# 2) Bitta dalada Birlamchidan keyingi ekin (Ikkilamchi, shu jumladan "aniqlanmagan", yoki Mavsumiy) —
#    Birlamchi tugagan oydan keyin boshlanadi va 10-oygacha: Birlamchi 3-5 → 6-10, 3-6 → 7-10.
# 3) Mavsumiy (Bog', Paxta, Boshqa ...) — 3-10; faqat ostida Birlamchi bo'lsa (2-qoida) kechroq boshlanadi.
#    Ozuqa yoki Bug'doy ustidagi Paxta — Ikkilamchi ekin (paxta_ikkilamchi, quyida).
# 4) Birlamchisi topilmagan Ikkilamchi — 7-10.
#
# Qidirish tartibi: zona → viloyat → hammasi; har birida avval ekin turi, keyin "_boshqa".
# Zona nomlari config/zonalar.yaml dagidek. Ekin nomi `turi` ustunidagidek (apostrof ').

hammasi:
  Birlamchi ekin:
    _boshqa: [3, 6]                 # Bug'doy, Makka, Kartoshka, Loviya

## 2. Konturlarni o'qish

In [3]:
df, mc = suv.oqi(KIRISH)               # atributlar
G = suv.geom_oqi(KIRISH, mc)           # mavsumli konturlar geometriyasi
print(f"{len(df)} kontur, mavsum ustuni: {mc}")
for x in suv.nomlarni_tekshir(df, cfg, zm):
    print("OGOHLANTIRISH:", x)
df.groupby(["mavsum", "suv_boshi", "suv_oxiri"], dropna=False).size().rename("kontur (hozir)")

533060 kontur, mavsum ustuni: season


mavsum           suv_boshi  suv_oxiri
Birlamchi ekin   3          4               447
                            5             33097
                            6             82471
Ikkilamchi ekin  5          10              107
                 6          10            34985
                 7          10            84286
Mavsumiy ekin    3          10           297204
                 5          10              463
Name: kontur (hozir), dtype: int64

## 3. Hisob: Paxta → Ikkilamchi va suv oylari

In [4]:
df, L = suv.hisobla(df, G, cfg, zm)
d = df[df.mavsum.notna()].assign(yangi=lambda x: [suv.oy_matn(b, o) for b, o in zip(x.b, x.o)])
(d.groupby(["yangi_mavsum", "yangi"]).agg(kontur=("uniqueid", "size"), maydon_ga=("maydon", "sum")).round().astype(int))

Paxta ['Ozuqa', "Bug'doy"] ustida → Ikkilamchi ekin: 463 kontur (7,184 ga); ostidagi Birlamchi: 447 kontur
Ikkilamchi ekin: 119840 / 119841 kontur Birlamchidan keyin — boshi: {5: 570, 6: 34985, 7: 84285}


kontur  maydon_ga
yangi_mavsum    yangi                   
Birlamchi ekin  3-4       447       7025
                3-5     33097     230872
                3-6     82471     365599
Ikkilamchi ekin 5-10      570       7298
                6-10    34985     230847
                7-10    84286     365562
Mavsumiy ekin   3-10   297204    1176324

### 3a. Ikkilamchiga o'tgan Paxta konturlari (viloyat / tuman)

In [5]:
px = df[df.mavsum != df.yangi_mavsum]
print(f"{len(px)} kontur, {px.maydon.sum():,.0f} ga")
px.groupby(["viloyat", "tuman"]).agg(kontur=("uniqueid", "size"), maydon_ga=("maydon", "sum")).round().sort_values("kontur", ascending=False)

463 kontur, 7,184 ga


kontur  maydon_ga
viloyat              tuman                                  
Sirdaryo viloyati    Sardoba tumani           120     3061.0
                     Oqoltin tumani            91     2427.0
Samarqand viloyati   Pastdarg'om tumani        22      136.0
Qashqadaryo viloyati Chiroqchi tumani          19       52.0
                     Shahrisabz tumani         18       84.0
                     G'uzor tumani             15      141.0
                     Koson tumani              14       65.0
Farg'ona viloyati    Buvayda tumani            14       52.0
Qashqadaryo viloyati Qarshi tumani             13      155.0
                     Qamashi tumani            10       58.0
Farg'ona viloyati    Dang'ara tumani           10       39.0
Samarqand viloyati   Oqdaryo tumani             9       46.0
Sirdaryo viloyati    Sirdaryo tumani            9       62.0
Qashqadaryo viloyati Kasbi tumani               8      190.0
Sirdaryo viloyati    Mirzaobod tumani           7       49.0
Qashqadaryo viloyati Yakkabog' tumani           7       65.0
                     Ko'kdala tumani            7       54.0
Farg'ona viloyati    Farg'ona tumani            7       62.0
Samarqand viloyati   Ishtixon tumani            6       24.0
Qashqadaryo viloyati Mirishkor tumani           6       35.0
Farg'ona viloyati    Uchko'prik tumani          5        9.0
Sirdaryo viloyati    Sayxunobod tumani          5       41.0
Farg'ona viloyati    Qo'shtepa tumani           4       29.0
                     Furqat tumani              4        9.0
                     Beshariq tumani            4       19.0
Sirdaryo viloyati    Boyovut tumani             4       50.0
Farg'ona viloyati    Bog'dod tumani             3       17.0
Qashqadaryo viloyati Nishon tumani              3       16.0
Samarqand viloyati   Kattaqo'rg'on tumani       3       14.0
                     Payariq tumani             3        8.0
Sirdaryo viloyati    Guliston tumani            3       23.0
                     Xovos tumani               3       48.0
Samarqand viloyati   Narpay tumani              2        5.0
Farg'ona viloyati    Yozyovon tumani            1       13.0
Qashqadaryo viloyati Muborak tumani             1       10.0
Farg'ona viloyati    O'zbekiston tumani         1        7.0
                     Toshloq tumani             1        7.0
Samarqand viloyati   Paxtachi tumani            1        2.0

## 4. O'zgarish jadvali (eski → yangi) — CSV ga

In [6]:
T = suv.ozgarish_jadvali(df)
out_csv = os.path.join(CHIQISH_PAPKA, NOM + "_ozgarish.csv")
T.to_csv(out_csv, index=False, encoding="utf-8-sig"); print("->", out_csv)
T[T.maydon_ga >= 500]

-> d:\Tasks\bir_ikki_lamchi\natija\ekin_yerlar_qisqa_v2_fenologiya_v3_fenologiya_ozgarish.csv


,viloyat,mavsum,yangi_mavsum,turi,eski,yangi,kontur,maydon_ga
2,Farg'ona viloyati,Birlamchi ekin,Birlamchi ekin,Bug'doy,3-6,3-6,18221,91748
0,Farg'ona viloyati,Birlamchi ekin,Birlamchi ekin,Birlamchi ekin aniqlanmagan,3-6,3-6,4383,13246
3,Farg'ona viloyati,Birlamchi ekin,Birlamchi ekin,Makka,3-6,3-6,1529,2601
5,Farg'ona viloyati,Birlamchi ekin,Birlamchi ekin,Ozuqa,3-5,3-5,579,2194
12,Farg'ona viloyati,Ikkilamchi ekin,Ikkilamchi ekin,Makka,7-10,7-10,10525,47312
10,Farg'ona viloyati,Ikkilamchi ekin,Ikkilamchi ekin,Ikkilamchi ekin aniqlanmagan,7-10,7-10,9444,35854
13,Farg'ona viloyati,Ikkilamchi ekin,Ikkilamchi ekin,Mosh,7-10,7-10,3345,16668
17,Farg'ona viloyati,Ikkilamchi ekin,Ikkilamchi ekin,Sholi,7-10,7-10,1075,4893
15,Farg'ona viloyati,Ikkilamchi ekin,Ikkilamchi ekin,Sabzi,7-10,7-10,358,2855
9,Farg'ona viloyati,Ikkilamchi ekin,Ikkilamchi ekin,Ikkilamchi ekin aniqlanmagan,6-10,6-10,531,2016


## 5. GPKG yozish (~1 daqiqa, ~400 MB nusxa)

In [7]:
h = suv.gpkg_yangila(KIRISH, CHIQISH, df, mc)
print("->", CHIQISH); print("yangilangan qatorlar:", h)

-> d:\Tasks\bir_ikki_lamchi\natija\ekin_yerlar_qisqa_v2_fenologiya_v3_fenologiya.gpkg
yangilangan qatorlar: {'ekin_yerlar': 533060}


## 6. Tekshiruv: suv / season dan boshqa hech narsa o'zgarmaganmi (geometriya ham)

In [8]:
farq = suv.faqat_suv_farqi(KIRISH, CHIQISH)
print("suv_boshi/suv_oxiri/season dan boshqa farq qilgan qatorlar:", farq)
assert all(v == 0 for v in farq.values()), "DIQQAT: boshqa ustunlar o'zgargan!"
v3, _ = suv.oqi(CHIQISH)
v3.groupby(["mavsum", "suv_boshi", "suv_oxiri"], dropna=False).size().rename("kontur (v3)")

suv_boshi/suv_oxiri/season dan boshqa farq qilgan qatorlar: {'ekin_yerlar': 0}


mavsum           suv_boshi  suv_oxiri
Birlamchi ekin   3          4               447
                            5             33097
                            6             82471
Ikkilamchi ekin  5          10              570
                 6          10            34985
                 7          10            84286
Mavsumiy ekin    3          10           297204
Name: kontur (v3), dtype: int64

## 7. Birlamchi va Ikkilamchi konturlar chegarasi — bir xil / farqli

v3 fayl bo'yicha (Ikkilamchiga o'tgan Paxta ham kiradi). Har kontur uchun qarama-qarshi mavsumdagi eng mos juft:
- **aynan bir xil** — polygon chegarasi to'liq teng;
- **deyarli bir xil** — IoU ≥ 0.99 (farq < 1%, chegarada mayda tafovut);
- **farq qiladi** — bitta dalada (kesishma ≥ 50%), lekin chegara boshqa (IoU < 0.99);
- **jufti yo'q** — qarama-qarshi mavsumda ≥ 50% kesishadigan kontur yo'q.

In [9]:
G3 = suv.geom_oqi(CHIQISH, mc)
C = suv.chegara_solishtir(G3)
pd.crosstab(C.tomon, C.toifa, margins=True, margins_name="Jami")

toifa,aynan bir xil,deyarli bir xil,farq qiladi,jufti yo'q,Jami
tomon,,,,,
Birlamchi,105201,3609,7205,0,116015
Ikkilamchi,105201,3609,11030,1,119841
Jami,210402,7218,18235,1,235856


In [10]:
# viloyat bo'yicha
pd.crosstab([C.tomon, C.viloyat], C.toifa, margins=True, margins_name="Jami")

toifa                            aynan bir xil  deyarli bir xil  farq qiladi  jufti yo'q    Jami
tomon      viloyat                                                                              
Birlamchi  Farg'ona viloyati             23538              362          870           0   24770
           Qashqadaryo viloyati          29433              726         3001           0   33160
           Samarqand viloyati            43816             1949         1556           0   47321
           Sirdaryo viloyati              8414              572         1778           0   10764
Ikkilamchi Farg'ona viloyati             23538              362         1489           0   25389
           Qashqadaryo viloyati          29433              726         4680           0   34839
           Samarqand viloyati            43816             1949         3072           0   48837
           Sirdaryo viloyati              8414              572         1789           1   10776
Jami                                    210402             7218        18235           1  235856

In [11]:
# bitta konturga nechta juft to'g'ri keladi (1 dan ko'p — dala bo'lingan)
C.groupby("tomon").juft_soni.value_counts().unstack(fill_value=0)

juft_soni,0,1,2,3,4,5,6,7
tomon,,,,,,,,
Birlamchi,0,111495,3776,635,87,11,9,2
Ikkilamchi,1,118517,1109,176,33,3,1,1


In [12]:
# farq qiladigan va jufti yo'q konturlar ro'yxati — CSV (ArcGIS da uniqueid bo'yicha topish uchun)
out = os.path.join(CHIQISH_PAPKA, NOM + "_chegara_farqli.csv")
C[C.toifa.isin(["farq qiladi", "jufti yo'q"])].to_csv(out, index=False, encoding="utf-8-sig")
print("->", out)
C[C.toifa == "farq qiladi"].groupby("tomon").iou.describe().round(2)

-> d:\Tasks\bir_ikki_lamchi\natija\ekin_yerlar_qisqa_v2_fenologiya_v3_fenologiya_chegara_farqli.csv


,count,mean,std,min,25%,50%,75%,max
tomon,,,,,,,,
Birlamchi,7205.0,0.59,0.23,0.0,0.47,0.61,0.76,0.99
Ikkilamchi,11030.0,0.47,0.26,0.0,0.26,0.48,0.67,0.99
